In [ ]:
def bagging_het(X_train, y_train, T, estimators, X_test):
    """
    Crea un modelo Bagging usando estimadores heterogéneos.
    En que genera una cantidad T de muestras bootstrap y
    para cada una de ellas se entrena un estimador escogido 
    en forma aleatoria desde la bolsa disponible. 
    La estimación final usando los modelos entrenados se
    realiza por mayoría de votos.

    Basado en 

    Parámetros:
    -----------
    X_train : DataFrame
        Conjunto de entrenamiento sin columna target.
    y_train : Series
        Datos con las clases asociadas a cada obs. de X_train
    T : int
        Representa la cantidad de muestras bootstrap a generar,
        equivalente a la cantidad de estimadores a entrenar.
    X_test : DataFrame
        Conjunto de test sobre el cual se somete el ensamble        

    Retorna:
    --------
    list
        Lista con los estimadores entrenados

    np.array
        Array con cantidad de filas igual a la cantidad de 
        observaciones de X_test y T columnas cada una con
        la clasificación predicha por el estimador asignado

    Series
        resultado por mayoría de votos para el conjunto de test

    list
        Lista con los índices no repetidos de las observaciones
        excluidad en cada muestra bootstrap
    """
    # Esta lista almacenera los estimadores entrenados en cada iteracion
    trained_model = []

    
    yhat_test = np.zeros((X_test.shape[0], T))

    # Lista que acumulara los indices de las observaciones que se quedaron fuera de las muestras bootstrap (conocidas como muestras out-of-bag u OOB)
    idx_oob = []

    
    for t in np.arange(0, T):

        
        sa1 = X_train.sample(n=X_train.shape[0], replace=True)

        idx_oob = list(set(idx_oob + list(set(X_train.index)-set(sa1.index))))
        
        idx_estimator = np.random.randint(0, len(estimators))
        estimator = estimators[idx_estimator]
        #print(idx_estimator, end='; ')
        
        estimator.fit(sa1, y_train[sa1.index])
        trained_model.append(estimator)

        yhat_test[:,t] = estimator.predict(X_test)
     
    yhat_out = pd.Series(data=mode(yhat_test, axis=1)[0], name='yhat')
        
    return trained_model, yhat_test, yhat_out, idx_oob


estimators = [
    LogisticRegression(random_state=42, max_iter=400), # se esta colocando un max_iter = 400 para que pueda converger
    DecisionTreeClassifier(random_state=42),
    SVC(probability=True, random_state=42, kernel='rbf'),
    SVC(probability=True, random_state=42, kernel='sigmoid')  
]

f1_scores = []

for estimator in estimators:
    estimator.fit(X_train_smote, y_train_smote)
    y_pred = estimator.predict(x_test)  
    score = f1_score(y_test, y_pred)  
    f1_scores.append(score)


for i, score in enumerate(f1_scores):
    print(f"F1-score por modelo {i} ({type(estimators[i]).__name__}): {score}")

best_model_index = np.argmax(f1_scores)

estimators_rep = estimators[:] 
best_model = estimators[best_model_index]
estimators_rep.extend([best_model] * 2) # se repite el mejor modelo

T= 200
trained_models, yhat_test, yhat_out, idx_oob = bagging_het(X_train_smote, y_train_smote, T, estimators, x_test)